In [ ]:
from __future__ import annotations
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Dict, List, Optional, Any, Union, Annotated, Literal
import random
import yaml
from collections import defaultdict, deque
from pathlib import Path
#import scoop

class BaseDistribution(BaseModel):
    """Base class for all distribution types."""
    name: Optional[str] = None

    def sample(self) -> Any:
        raise NotImplementedError
    

class NumericDistribution(BaseDistribution):
    
    """
    Default Sampling logic:
    - If sd is provided → Gaussian(mean, sd) + clipping
    - If sd is missing or zero → Uniform(min_value, max_value)
    """

    mean: float = 0
    sd: Optional[float] = None
    min_value: Optional[float] = None
    max_value: Optional[float] = None
    as_int: bool = False

    def sample(self):
        # Uniform if no sd
        if not self.sd:
            if self.min_value is None or self.max_value is None:
                raise ValueError(f"Uniform sampling requires min_value and max_value: {self.name}")
            val = random.uniform(self.min_value, self.max_value)
        else:
            val = random.gauss(self.mean, self.sd)

        # clip
        if self.min_value is not None:
            val = max(self.min_value, val)
        if self.max_value is not None:
            val = min(self.max_value, val)

        return int(val) if self.as_int else val


class DiscreteDistribution(BaseDistribution):
    """Discrete labelled or numeric distribution - simple lookup"""

    probs: Dict[Union[int, float, str], float] = Field(
        ..., description="Mapping of label → probability (must sum to 1.0)"
    )

    @field_validator("probs")
    @classmethod
    def validate_probs(cls, v: Dict[Union[int, float, str], float]):
        total = sum(v.values())
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"Probabilities must sum to 1 (got {total}) for {cls.name}")
        return v

    def sample(self) -> Union[int, float, str]:
        vals, weights = zip(*self.probs.items())
        return random.choices(vals, weights=weights, k=1)[0] 


class GraphEdge(BaseModel):
    from_: str = Field(..., alias="from")
    to: str
    using: str
    table: Dict[Any, List[float]]


class ConditionalDiscreteDistribution(BaseDistribution):
    """
    A distribution that depends on parent node values.

    It doesn't hold a single fixed prob vector, but rather:
    - labels: ordered list of outcome labels for the child node
    - incoming_edges: GraphEdges with CPT rows

    At sample time, we call sample_given(parent_values) to:
    - look up each parent’s row in its table
    - combine rows multiplicatively
    - normalize
    - draw a sample
    """

    labels: List[str]
    incoming_edges: List[GraphEdge]

    def get_probs(self, parent_values: Dict[str, Any]) -> Dict[str, float]:
        n = len(self.labels)
        probs = [1.0] * n  # neutral element for multiplicative fusion

        for edge in self.incoming_edges:
            parent_value = parent_values.get(edge.from_)

            row = edge.table.get(parent_value, edge.table.get("default"))
            if row is None:
                raise ValueError(
                    f"No row in CPT for parent={edge.from_}, value={parent_value}"
                )
            if len(row) != n:
                raise ValueError(
                    f"CPT row length ({len(row)}) does not match number of labels ({n} in node {self.name})"
                    f"for child of edge {edge.from_} -> {edge.to}"
                )

            probs = [a * b for a, b in zip(probs, row)]

        total = sum(probs)
        if total <= 0:
            raise ValueError(
                "Conditional probabilities collapsed to zero. "
                "Check CPTs and parent assignments."
            )

        probs = [p / total for p in probs]
        return dict(zip(self.labels, probs))

    def sample_given(self, parent_values: Dict[str, Any]) -> Any:
        probs = self.get_probs(parent_values)
        vals, weights = zip(*probs.items())
        return random.choices(vals, weights=weights, k=1)[0]

    def sample(self) -> Any:
        # Not meaningful without parent values
        raise RuntimeError("Use sample_given(parent_values) for conditional distributions.")

class BaseGraphNode(BaseModel):
    """Base class for all graph nodes."""

    name: Optional[str] = None
    type: str
    built_distribution: Optional[BaseDistribution] = None

    def sample(self, parent_values: Optional[Dict[str, Any]] = None) -> Any:
        if not self.built_distribution:
            raise ValueError(f"Distribution not built yet for node {self.name}")

        if isinstance(self.built_distribution, ConditionalDiscreteDistribution):
            if parent_values is None:
                raise ValueError(
                    f"Node {self.name} has conditional distribution; parent values required."
                )
            return self.built_distribution.sample_given(parent_values)
        else:
            return self.built_distribution.sample()


class NumericNode(BaseGraphNode):
    type: Literal["numeric"]

    mean: float = 0
    sd: Optional[float] = None
    min_value: Optional[float] = None
    max_value: Optional[float] = None
    as_int: bool = False

    @field_validator("sd")
    @classmethod
    def sd_positive(cls, v):
        if v is not None and v <= 0:
            raise ValueError("'sd' must be positive.")
        return v

    @model_validator(mode="after")
    def build_numeric_distribution(self):
        self.built_distribution = NumericDistribution(
            mean=self.mean,
            sd=self.sd,
            min_value=self.min_value,
            max_value=self.max_value,
            as_int=self.as_int,
        )
        return self


class CategoricalNode(BaseGraphNode):
    type: Literal["categorical"]

    # For unconditional nodes: user can specify distribution explicitly
    # For conditional nodes: distribution may be empty and labels provided instead
    distribution: Dict[Union[str, float, int], float] = Field(default_factory=dict)
    labels: Optional[List[str]] = None


GraphNode = Annotated[
    Union[NumericNode, CategoricalNode],
    Field(discriminator="type"),
]


class TransformationGraph(BaseModel):
    """
    A directed acyclic graph of GraphNodes and GraphEdges.
    Handles:
    - building adjacency
    - topological ordering
    - assigning correct distribution types (unconditional vs conditional)
    - sampling in dependency order
    """

    nodes: Dict[str, GraphNode]
    edges: List[GraphEdge]

    order: List[str] = Field(default_factory=list)
    graph: Dict[str, List[str]] = Field(default_factory=dict)

    @model_validator(mode="after")
    def build_graph_and_order(self):
        # 1. Assign names to nodes based on dict keys
        for name, node in self.nodes.items():
            node.name = name
            if node.built_distribution:
                node.built_distribution.name = name 

        # 2. Build adjacency map
        g = defaultdict(list)
        for edge in self.edges:
            g[edge.from_].append(edge.to)
        self.graph = dict(g)

        # 3. Topological sort
        self.order = self._topological_sort()

        # 4. Assign distribution types
        for name, node in self.nodes.items():
            incoming = [e for e in self.edges if e.to == name]

            # Numeric nodes already built their distributions in model validator
            if isinstance(node, NumericNode):
                continue

            if isinstance(node, CategoricalNode):
                if incoming:
                    # Conditional categorical node
                    # Decide label order: prefer explicit labels, else from any CPT row length
                    if node.labels:
                        labels = node.labels
                    elif node.distribution:
                        labels = list(node.distribution.keys())
                    else:
                        # Try infer from first edge row
                        first_edge = incoming[0]
                        some_row = next(iter(first_edge.table.values()))
                        if node.labels and len(node.labels) != len(some_row):
                            raise ValueError(
                                f"Label count for node {name} does not match CPT row length."
                            )
                        if node.labels:
                            labels = node.labels
                        else:
                            # create dummy labels if absolutely necessary
                            labels = [f"cat_{i}" for i in range(len(some_row))]

                    node.built_distribution = ConditionalDiscreteDistribution(
                        labels=labels,
                        incoming_edges=incoming,
                    )
                else:
                    # Unconditional categorical node
                    if not node.distribution:
                        # If we have labels but no probs, use equal probabilities
                        if node.labels:
                            n = len(node.labels)
                            node.distribution = {label: 1.0 / n for label in node.labels}
                        else:
                            raise ValueError(
                                f"Unconditional categorical node '{name}' "
                                f"must have either 'distribution' or 'labels'."
                            )

                    node.built_distribution = DiscreteDistribution(
                        probs=node.distribution
                    )
        return self

    def _topological_sort(self) -> List[str]:
        indegree = {n: 0 for n in self.nodes}
        for e in self.edges:
            indegree[e.to] += 1

        q = deque([n for n, d in indegree.items() if d == 0])
        result: List[str] = []

        while q:
            n = q.popleft()
            result.append(n)
            for nxt in self.graph.get(n, []):
                indegree[nxt] -= 1
                if indegree[nxt] == 0:
                    q.append(nxt)

        if len(result) != len(self.nodes):
            raise ValueError("Graph contains a cycle")

        return result

    def sample_one(self) -> Dict[str, Any]:
        """
        Sample a single joint assignment for all nodes,
        respecting dependency order and conditional distributions.
        """
        sampled: Dict[str, Any] = {}

        for name in self.order:
            node = self.nodes[name]

            if isinstance(node.built_distribution, ConditionalDiscreteDistribution):
                sampled[name] = node.sample(parent_values=sampled)
            else:
                sampled[name] = node.sample()

        return sampled

    def sample_many(self, n: int) -> List[Dict[str, Any]]:
        """Convenience: draw n independent samples."""
        return [self.sample_one() for _ in range(n)]


def load_transformation_graph(yaml_path: str) -> TransformationGraph:
    """
    Load the YAML graph into a TransformationGraph,
    build a DAG, validate, and return a GraphEngine instance.
    """
    with open(yaml_path, 'r') as f:
        raw = yaml.safe_load(f)

    tg = TransformationGraph.model_validate(raw['graph'])#(**raw['graph'])
    return tg


def make_mermaid(graph: TransformationGraph) -> str:
    """
    Generate a Mermaid flowchart from a TransformationGraph.

    Features:
    - Disconnected nodes first, then connected (topologically sorted)
    - Subgraphs by prefix (segment before '.')
    - Color coding:
        numeric = blue
        categorical = green
    - Edge labels show 'using' and a compact CPT summary
    """

    nodes = graph.nodes
    edges = graph.edges

    def mermaid_id(name: str) -> str:
        return name.replace(".", "_").replace("-", "_")

    def get_group(name: str) -> str:
        return name.split(".")[0] if "." in name else "root"

    # Determine connected vs disconnected
    connected = set()
    for e in edges:
        connected.add(e.from_)
        connected.add(e.to)

    all_nodes = set(nodes.keys())
    disconnected = sorted(all_nodes - connected)

    # Topological part for connected nodes
    connected_sorted = [n for n in graph.order if n in connected]

    ordered_nodes = disconnected + connected_sorted

    # Group by prefix
    groups: Dict[str, List[str]] = {}
    for name in ordered_nodes:
        grp = get_group(name)
        groups.setdefault(grp, []).append(name)

    lines: List[str] = ["flowchart LR\n"]

    # Render groups and nodes
    for group_name, group_nodes in groups.items():
        if group_name != "root":
            lines.append(f"    subgraph {group_name}")
        else:
            lines.append("    %% root nodes (no prefix)")

        for name in group_nodes:
            node = nodes[name]
            node_id = mermaid_id(name)

            # Node summary + color
            summary = ""
            if isinstance(node, NumericNode):
                summary = f"<br/>N({node.mean}, {node.sd})"
                color = "#A3C4FF"  # soft blue
            elif isinstance(node, CategoricalNode):
                if isinstance(node.built_distribution, DiscreteDistribution):
                    cats = list(node.built_distribution.probs.keys())
                    if len(cats) <= 4:
                        summary = "<br/>" + ", ".join(map(str, cats))
                    else:
                        summary = f"<br/>{len(cats)} categories"
                elif isinstance(node.built_distribution, ConditionalDiscreteDistribution):
                    summary = "<br/>conditional"
                else:
                    summary = ""
                color = "#C3F9C3"  # soft green
            else:
                color = "#EEE"

            lines.append(
                f'        {node_id}["{name}{summary}"]:::coloredNode_{node_id}'
            )
            lines.append(
                f'        classDef coloredNode_{node_id} fill:{color},stroke:#333,stroke-width:1px;'
            )

        if group_name != "root":
            lines.append("    end\n")
        else:
            lines.append("")

    # Render edges
    for edge in edges:
        src = mermaid_id(edge.from_)
        dst = mermaid_id(edge.to)

        # Compact tooltip for CPT
        keys = list(edge.table.keys())
        if len(keys) > 3:
            tooltip = f"{len(keys)} rows"
        else:
            tooltip = ", ".join(map(str, keys))

        label = edge.using
        if tooltip:
            label += f" #40;{tooltip}#41;"

        lines.append(f'    {src} -->|{label}| {dst}')

    return "\n".join(lines)

In [1]:
from pathlib import Path
from scoop.utils import load_transformation_graph, make_mermaid
from scoop.configuration import transformation_graph

In [2]:
transformation_graph

TransformationGraph(nodes={'person.age': NumericNode(name='person.age', type='numeric', built_distribution=NumericDistribution(name='person.age', mean=70.0, sd=7.0, min_value=50.0, max_value=95.0, as_int=True), mean=70.0, sd=7.0, min_value=50.0, max_value=95.0, as_int=True), 'person.age_group': NumericRangeNode(name='person.age_group', type='numeric_range', built_distribution=None, from_='person.age', ranges=['<60', '60-69', '70-79', '80+']), 'person.year_of_diagnosis': NumericNode(name='person.year_of_diagnosis', type='numeric', built_distribution=NumericDistribution(name='person.year_of_diagnosis', mean=0.0, sd=None, min_value=2015.0, max_value=2020.0, as_int=True), mean=0, sd=None, min_value=2015.0, max_value=2020.0, as_int=True), 'person.years_since_diagnosis': NumericNode(name='person.years_since_diagnosis', type='numeric', built_distribution=NumericDistribution(name='person.years_since_diagnosis', mean=0.0, sd=None, min_value=0.0, max_value=3.0, as_int=True), mean=0, sd=None, min

In [3]:
# graph = load_transformation_graph(config_path / "probabilistic_transformations.yaml")
print(transformation_graph.order)          # dependency order
print(transformation_graph.nodes)          # correct node types
print(transformation_graph.edges)          # CPT edges
print(transformation_graph.sample_one())   # fully conditional sampling


['person.age', 'person.age_group', 'person.year_of_diagnosis', 'person.years_since_diagnosis', 'person.sex', 'person.smoking_history', 'person.variant', 'person.weight', 'condition_occurrence.laterality', 'person.performance_status', 'condition_occurrence.cancer_stage', 'condition_occurrence.morphology', 'condition_occurrence.lobe', 'treatment_episode.radiotherapy_eligibility', 'treatment_episode.operability', 'treatment_episode.regimen_selection']
{'person.age': NumericNode(name='person.age', type='numeric', built_distribution=NumericDistribution(name='person.age', mean=70.0, sd=7.0, min_value=50.0, max_value=95.0, as_int=True), mean=70.0, sd=7.0, min_value=50.0, max_value=95.0, as_int=True), 'person.age_group': NumericRangeNode(name='person.age_group', type='numeric_range', built_distribution=None, from_='person.age', ranges=['<60', '60-69', '70-79', '80+']), 'person.year_of_diagnosis': NumericNode(name='person.year_of_diagnosis', type='numeric', built_distribution=NumericDistributio

In [ ]:
print(make_mermaid(graph))